<h1 style="color:DodgerBlue">Индивидальный проект</h1>

<h2 style="color:DodgerBlue">Название проекта: Система управления задачами проекта</h2>

----

### Вариант задания 23


<h2 style="color:DodgerBlue">Описание проекта:</h2>

----

Проект представляет собой систему управления задачами.

Базовый класс Task содержит общие данные задачи:
- идентификатор задачи (TaskId);
- название (TaskName);
- приоритет (Priority);
- исполнитель (AssignedTo).

От класса Task наследуются:
- DelegateTask — делегированная задача;
- TeamTask — командная задача;
- ResearchTask — исследовательская задача.

Каждый производный класс добавляет собственные свойства и методы и переопределяет отдельные методы базового класса.

#### Дополнительное задание
Добавьте к существующим классам (базовому и производным 3-4 атрибута и метода) используйте в проекте коллекции, делегаты, события.


<h2 style="color:DodgerBlue">Реализация:</h2>

----

In [ ]:
using System;
using System.Collections.Generic;
using System.Linq;

public delegate void TaskEventHandler(Task task, string message);

public class Task
{
    public int TaskId { get; set; }
    public string TaskName { get; set; }
    public string Priority { get; set; }
    public string AssignedTo { get; set; }

    public DateTime CreatedAt { get; set; }
    public bool IsCompleted { get; set; }
    public int EstimatedHours { get; set; }

    public event TaskEventHandler OnCompleted;
    public event TaskEventHandler OnAssigned;

    public Task(int taskId, string taskName, string priority, int estimatedHours = 8)
    {
        TaskId = taskId;
        TaskName = taskName;
        Priority = priority;
        AssignedTo = "Не назначен";
        CreatedAt = DateTime.Now;
        IsCompleted = false;
        EstimatedHours = estimatedHours;
    }

    public virtual void MarkAsComplete()
    {
        IsCompleted = true;
        Console.WriteLine($"Задача "{TaskName}" выполнена.");
        OnCompleted?.Invoke(this, $"Задача #{TaskId} успешно завершена исполнитель: {AssignedTo}");
    }

    public virtual void GetTaskDetails()
    {
        Console.WriteLine($"ID: {TaskId} | Название: {TaskName} | Приоритет: {Priority} | Исполнитель: {AssignedTo} | Статус: {(IsCompleted ? "Завершена" : "В работе")}");
    }

    public virtual void ReassignTo(string person)
    {
        string oldPerson = AssignedTo;
        AssignedTo = person;
        Console.WriteLine($"Задача "{TaskName}" назначена: {person}");
        OnAssigned?.Invoke(this, $"Задача #{TaskId} переназначена с '{oldPerson}' на '{person}'");
    }

    public void ReassignTo(string person, string reason)
    {
        ReassignTo(person);
        Console.WriteLine($"  Причина смены исполнителя: {reason}");
    }
}

public class DelegateTask : Task
{
    public DateTime DueDate { get; set; }
    public string DelegatedBy { get; set; }
    public string ApprovalStatus { get; set; }

    public DelegateTask(int taskId, string taskName, string priority, DateTime dueDate, string delegatedBy, int estimatedHours = 12)
        : base(taskId, taskName, priority, estimatedHours)
    {
        DueDate = dueDate;
        DelegatedBy = delegatedBy;
        ApprovalStatus = "Ожидает согласования";
    }

    public override void MarkAsComplete()
    {
        base.MarkAsComplete();
        ApprovalStatus = "Утверждено";
        Console.WriteLine($"Делегированная задача согласована: {DelegatedBy}. Срок: {DueDate:d}");
    }

    public void ShowDeadline()
    {
        Console.WriteLine($"Срок выполнения задачи "{TaskName}": {DueDate:d}");
    }

    public void ExtendDeadline(int days)
    {
        DueDate = DueDate.AddDays(days);
        Console.WriteLine($"Срок продлен на {days} дн. Новый срок: {DueDate:d}");
    }
}

public class TeamTask : Task
{
    public string TeamName { get; set; }
    public int MemberCount { get; set; }
    public int SprintNumber { get; set; }

    public TeamTask(int taskId, string taskName, string priority, string teamName, int memberCount, int sprintNumber = 5)
        : base(taskId, taskName, priority, 40)
    {
        TeamName = teamName;
        MemberCount = memberCount;
        SprintNumber = sprintNumber;
    }

    public override void ReassignTo(string person)
    {
        base.ReassignTo(person);
        Console.WriteLine($"Командная задача "{TaskName}" закреплена за тимлидом команды "{TeamName}"");
    }

    public void AddMember()
    {
        MemberCount++;
        Console.WriteLine($"В команду "{TeamName}" добавлен специалист. Всего: {MemberCount}");
    }

    public void ChangeSprint(int sprint)
    {
        SprintNumber = sprint;
        Console.WriteLine($"Задача перемещена в спринт #{SprintNumber}");
    }
}

public class ResearchTask : Task
{
    public string DataSource { get; set; }
    public string ResearchMethod { get; set; }
    public string Findings { get; set; }

    public ResearchTask(int taskId, string taskName, string priority, string dataSource, string researchMethod)
        : base(taskId, taskName, priority, 20)
    {
        DataSource = dataSource;
        ResearchMethod = researchMethod;
        Findings = "В процессе анализа";
    }

    public override void GetTaskDetails()
    {
        base.GetTaskDetails();
        Console.WriteLine($"  [Исследование] Источник: {DataSource} | Метод: {ResearchMethod} | Результаты: {Findings}");
    }

    public void AddFindings(string findings)
    {
        Findings = findings;
        Console.WriteLine($"Результаты исследования обновлены: {Findings}");
    }

    public void ChangeResearchMethod(string method)
    {
        ResearchMethod = method;
        Console.WriteLine($"Метод исследования изменен на: {ResearchMethod}");
    }
}

Console.WriteLine("================ СИСТЕМА УПРАВЛЕНИЯ ЗАДАЧАМИ (ВАРИАНТ 23) ================\n");

DelegateTask dt = new DelegateTask(1, "Подготовить финансовый отчет", "Высокий", DateTime.Today.AddDays(7), "Директор");
TeamTask tt = new TeamTask(2, "Интеграция платежного шлюза", "Критический", "Fintech Team", 4, 5);
ResearchTask rt = new ResearchTask(3, "Исследование производительности БД", "Средний", "PostgreSQL Benchmarks", "Нагрузочное тестирование");

Console.WriteLine("--- 1. События и обработчики (делегаты) ---");
dt.OnAssigned += (t, msg) => Console.WriteLine($"[СОБЫТИЕ НАЗНАЧЕНИЯ] {msg}");
dt.OnCompleted += (t, msg) => Console.WriteLine($"[СОБЫТИЕ ЗАВЕРШЕНИЯ] {msg}");

tt.OnAssigned += (t, msg) => Console.WriteLine($"[СОБЫТИЕ НАЗНАЧЕНИЯ] {msg}");
tt.OnCompleted += (t, msg) => Console.WriteLine($"[СОБЫТИЕ ЗАВЕРШЕНИЯ] {msg}");

dt.ReassignTo("Екатерина Смирнова", "Назначена ведущим финансистом");
dt.MarkAsComplete();

Console.WriteLine();
tt.ReassignTo("Дмитрий Волков");

Console.WriteLine();

Console.WriteLine("--- 2. Коллекции: List<Task>, Dictionary<int, Task>, HashSet<string> ---");

List<Task> projectTasks = new List<Task> { dt, tt, rt };

Dictionary<int, Task> taskDictionary = projectTasks.ToDictionary(t => t.TaskId);
if (taskDictionary.TryGetValue(2, out Task foundTask))
{
    Console.WriteLine($"Поиск по ID 2 в Dictionary:");
    foundTask.GetTaskDetails();
}

HashSet<string> teamMembers = new HashSet<string>();
foreach (var t in projectTasks)
{
    teamMembers.Add(t.AssignedTo);
}
teamMembers.Add("Дмитрий Волков");
Console.WriteLine($"\nУникальные участники в проекте (HashSet, всего {teamMembers.Count}): {string.Join(", ", teamMembers)}");

Console.WriteLine();

Console.WriteLine("--- 3. Лямбда-выражения и LINQ-запросы ---");

Console.WriteLine("Все задачи с приоритетом 'Высокий' или 'Критический':");
var highPriorityTasks = projectTasks.Where(t => t.Priority == "Высокий" || t.Priority == "Критический");
foreach (var t in highPriorityTasks)
{
    t.GetTaskDetails();
}

Console.WriteLine("\nЗадачи, требующие более 15 часов:");
var longTasks = projectTasks.Where(t => t.EstimatedHours > 15).Select(t => $"{t.TaskName} ({t.EstimatedHours} ч.)");
foreach (var item in longTasks)
{
    Console.WriteLine($" - {item}");
}
